In [ ]:
import json
from datasets import load_dataset
import pandas as pd
import sqlglot
from sqlglot import exp
import re

In [ ]:
DATASET_PATH = 'xlangai/spider'

In [ ]:
dataset = load_dataset(DATASET_PATH)

In [ ]:
statements = list(dataset['train']['query']) + list(dataset['validation']['query'])

In [ ]:
# remove unnecessary whitespace and newlines
for statement in statements:
    statement = statement.strip()

In [ ]:
print(statements[0])

In [ ]:
index = 0
keywords = []

In [ ]:
for index, statement in enumerate(statements):
    try:
        # Quan trọng: Nên chuẩn hóa khoảng trắng TRƯỚC khi parse để tránh lỗi ký tự lạ
        statement = re.sub(r'\s+', ' ', statement).strip()
        parsed = sqlglot.parse_one(statement)

        features = {
            "tables": [t.name.lower() for t in parsed.find_all(exp.Table)],
            
            # TRÍCH XUẤT CỘT TRONG SELECT
            "columns_select": [
                c.name.lower() 
                for c in parsed.find(exp.Select).find_all(exp.Column)
            ] if parsed.find(exp.Select) else [],

            # TRÍCH XUẤT CỘT TRONG JOIN ON
            "join_columns": [
                c.name.lower() 
                for j in parsed.find_all(exp.Join) 
                if j.args.get("on") 
                for c in j.args["on"].find_all(exp.Column)
            ],

            "columns_where": [c.name.lower() for c in parsed.find(exp.Where).find_all(exp.Column)] if parsed.find(exp.Where) else [],
            
            "columns_group": [c.name.lower() for c in parsed.find(exp.Group).find_all(exp.Column)] if parsed.find(exp.Group) else [],

            # TRÍCH XUẤT CỘT TRONG ORDER BY
            "columns_order": [
                c.name.lower() 
                for c in parsed.find(exp.Order).find_all(exp.Column)
            ] if parsed.find(exp.Order) else [],

            "join_types": [j.args.get("side") for j in parsed.find_all(exp.Join) if j.args.get("side")],
            "timestamp": index
        }
        keywords.append(features)

    except Exception as e:
        print(f"Error parsing statement at index {index}: {statement}")
        print(f"Detail: {e}")
        continue
    

In [ ]:
print(keywords[0:5])

In [ ]:
with open('preprocessed-data/spider_statements.json', 'w') as f:
    json.dump(keywords, f, indent=4)